# Load Public dataset - synthstrip

In [ ]:
!curl -O https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/data/SHA256
!curl -O https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/data/synthstrip_data_v1.5.tar
!curl -O https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/data/synthstrip_data_v1.5_2d.tar
!shasum -c SHA256
!tar -xf synthstrip_data_v1.5.tar
!tar -xf synthstrip_data_v1.5_2d.tar

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   185  100   185    0     0    164      0  0:00:01  0:00:01 --:--:--   164
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
 21 7008M   21 1494M    0     0   453k      0  4:23:54  0:56:15  3:27:39  483k0:00:59  4:32:02  480k    0   441k      0  4:31:03  0:01:04  4:29:59  479k65.2M    0     0   451k      0  4:24:45  0:02:27  4:22:18  485k:59  0:02:39  4:24:20  293k:29  0:04:05  4:17:24  481k   456k      0  4:22:12  0:05:00  4:17:12  241k 0   455k      0  4:22:47  0:05:26  4:17:21  484k5M    0     0   460k      0  4:19:43  0:07:58  4:11:45  483k    0  4:20:14  0:16:29  4:03:45  478k  466M    0     0   460k      0  4:19:40  0:17:16  4:02:24  480kk      0  4:19:28  0:17:35  4:01:53  482k    0  4:19:15  0:20:15  3:59:00  351

check that data is loaded

In [1]:
!curl -O -C - https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/data/SHA256
!curl -O -C - https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/data/synthstrip_data_v1.5.tar
!curl -O -C - https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/data/synthstrip_data_v1.5_2d.tar


** Resuming transfer from byte position 185
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100   314    0   314    0     0    338      0 --:--:-- --:--:-- --:--:--   338
** Resuming transfer from byte position 7349022720
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   314    0   314    0     0    602      0 --:--:-- --:--:-- --:--:--   602
** Resuming transfer from byte position 40427520
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   314    0   314    0     0    600      0 --:--:-- --:--:-- --:--:--   600


In [6]:
!ls -lh synthstrip_data_v1.5.tar
!shasum -c SHA256


-rw-rw-r-- 1 natalie natalie 6.9G Aug  9 14:00 synthstrip_data_v1.5.tar


synthstrip_data_v1.5_2d.tar: OK
synthstrip_data_v1.5.tar: OK


In [4]:
!curl -C - -O https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/data/synthstrip_data_v1.5.tar



** Resuming transfer from byte position 7349022720
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100   314    0   314    0     0    598      0 --:--:-- --:--:-- --:--:--   598


convert to npz

In [2]:
##
import os
import nibabel as nib
import numpy as np

data_dir = 'synthstrip_data_v1.5'
out_dir = 'converted_npz_new_3D'
os.makedirs(out_dir, exist_ok=True)

subject_dirs = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
for sub in subject_dirs:
    sub_path = os.path.join(data_dir, sub)
    img_path = os.path.join(sub_path, 'image.nii.gz')
    mask_path = os.path.join(sub_path, 'mask.nii.gz')
    if not (os.path.exists(img_path) and os.path.exists(mask_path)):
        continue
    img = nib.load(img_path).get_fdata().astype(np.float32)
    mask = nib.load(mask_path).get_fdata().astype(np.uint8)
    npz_name = f"{sub}.npz"
    np.savez_compressed(os.path.join(out_dir, npz_name), vol=img, seg=mask)

check what data has been downloaded

In [1]:
import os
import re
from collections import defaultdict

def get_info(dir):
    files = [f for f in os.listdir(dir) if f.endswith('.npz')]

    pattern = re.compile(r"([a-zA-Z0-9]+_[a-zA-Z0-9]+)_([a-zA-Z0-9]+)\.npz")

    type_counts = defaultdict(int)
    person_types = defaultdict(set)
    all_types = set()
    all_people = set()

    for fname in files:
        match = pattern.match(fname)
        if match:
            img_type = match.group(1)    # e.g. fsm_t1
            person = match.group(2)      # e.g. 24dz
            type_counts[img_type] += 1
            person_types[person].add(img_type)
            all_types.add(img_type)
            all_people.add(person)
        else:
            print(f"Skipping filename that doesn't match expected pattern: {fname}")

    # Then print counts as before
    print("Image count by type:")
    for t in sorted(type_counts):
        print(f"  {t}: {type_counts[t]}")

    print(f"\nNumber of unique people: {len(all_people)}")

    print("\nHow many image types does each patient have?")
    for person in sorted(person_types):
        print(f"  Patient {person}: {len(person_types[person])} types -> {person_types[person]}")

    from collections import Counter
    c = Counter(len(v) for v in person_types.values())
    print("\nPatient count by number of image types:")
    for n_types, n_patients in sorted(c.items()):
        print(f"  {n_patients} patients have {n_types} types")


NOTE: customization for local dir has to be done

In [4]:
get_info(dir = "converted_npz")

Image count by type:
  asl_epi: 19
  asl_t1: 7
  fsm_pd: 8
  fsm_qt1: 12
  fsm_t1: 10
  fsm_t2: 10
  infant_t1: 4
  ixi_dwi: 12
  ixi_mra: 11
  ixi_pd: 14
  ixi_t1: 13
  ixi_t2: 11
  qin_flair: 4
  qin_t1: 9
  qin_t2: 10

Number of unique people: 107

How many image types does each patient have?
  Patient 002: 2 types -> {'ixi_pd', 'ixi_dwi'}
  Patient 01: 1 types -> {'qin_flair'}
  Patient 013: 1 types -> {'ixi_pd'}
  Patient 015: 1 types -> {'ixi_pd'}
  Patient 017: 1 types -> {'ixi_t1'}
  Patient 019: 1 types -> {'ixi_pd'}
  Patient 021: 1 types -> {'ixi_mra'}
  Patient 022: 1 types -> {'ixi_t2'}
  Patient 02cd: 1 types -> {'fsm_t1'}
  Patient 03: 1 types -> {'qin_flair'}
  Patient 04: 1 types -> {'qin_t2'}
  Patient 05: 1 types -> {'qin_t2'}
  Patient 07tz: 1 types -> {'fsm_t2'}
  Patient 08: 2 types -> {'qin_t2', 'qin_t1'}
  Patient 09: 2 types -> {'qin_t2', 'qin_t1'}
  Patient 105: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 106: 1 types -> {'asl_epi'}
  Patient 10cv: 1 types -> {

In [3]:
get_info(dir = "converted_npz_new_2D")

Image count by type:
  asl_epi: 43
  asl_t1: 43
  fsm_pd: 32
  fsm_qt1: 32
  fsm_t1: 38
  fsm_t2: 36
  infant_t1: 16
  ixi_dwi: 32
  ixi_mra: 50
  ixi_pd: 50
  ixi_t1: 50
  ixi_t2: 50
  qin_flair: 17
  qin_t1: 54
  qin_t2: 39

Number of unique people: 179

How many image types does each patient have?
  Patient 002: 5 types -> {'ixi_t1', 'ixi_t2', 'ixi_mra', 'ixi_dwi', 'ixi_pd'}
  Patient 01: 3 types -> {'qin_t1', 'qin_t2', 'qin_flair'}
  Patient 012: 5 types -> {'ixi_t1', 'ixi_t2', 'ixi_mra', 'ixi_dwi', 'ixi_pd'}
  Patient 013: 5 types -> {'ixi_t1', 'ixi_t2', 'ixi_mra', 'ixi_dwi', 'ixi_pd'}
  Patient 015: 5 types -> {'ixi_t1', 'ixi_t2', 'ixi_mra', 'ixi_dwi', 'ixi_pd'}
  Patient 016: 4 types -> {'ixi_t1', 'ixi_pd', 'ixi_mra', 'ixi_t2'}
  Patient 017: 4 types -> {'ixi_t1', 'ixi_pd', 'ixi_mra', 'ixi_t2'}
  Patient 019: 4 types -> {'ixi_t1', 'ixi_pd', 'ixi_mra', 'ixi_t2'}
  Patient 01ob: 4 types -> {'fsm_t2', 'fsm_t1', 'fsm_qt1', 'fsm_pd'}
  Patient 02: 3 types -> {'qin_t1', 'qin_t2', 'qin

In [2]:
get_info(dir = "converted_npz_new_3D")

Image count by type:
  asl_epi: 43
  asl_t1: 43
  fsm_pd: 32
  fsm_qt1: 32
  fsm_t1: 38
  fsm_t2: 1

Number of unique people: 81

How many image types does each patient have?
  Patient 01ob: 4 types -> {'fsm_t2', 'fsm_t1', 'fsm_qt1', 'fsm_pd'}
  Patient 02cd: 3 types -> {'fsm_t1', 'fsm_qt1', 'fsm_pd'}
  Patient 07tz: 1 types -> {'fsm_t1'}
  Patient 08un: 3 types -> {'fsm_t1', 'fsm_qt1', 'fsm_pd'}
  Patient 101: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 105: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 106: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 109: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 10cv: 3 types -> {'fsm_t1', 'fsm_qt1', 'fsm_pd'}
  Patient 110: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 111: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 112: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 113: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 114: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 115: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 116: 2 types -> {'asl_epi', 'asl_t1'}
  Patient 117